## Notebook Workflow

### Obiettivi principali del notebook

1. **Caricamento e analisi del dataset**:
    - Importazione del file CSV contenente i metadati.
    - Analisi della distribuzione delle classi e dei fold.

2. **Preprocessing audio**:
    - Resampling di tutti i file audio a 22,050 Hz.
    - Normalizzazione a -1 dBFS.
    - Pad/trim per ottenere una lunghezza standard di 4 secondi.

3. **Estrazione delle feature**:
    - Generazione di log-mel-spettrogrammi con 128 bande mel.
    - Calcolo di 20 coefficienti MFCC.
    - Estrazione di centroidi spettrali e roll-off.

4. **Visualizzazione di esempio**:
    - Visualizzazione della forma d'onda e dello spettrogramma di un file audio processato.

5. **Processing e salvataggio dell'intero dataset**:
    - Mantenimento della struttura originale dei 10 fold.
    - Salvataggio dei file audio preprocessati e delle feature estratte.

### Struttura del dataset

- **File CSV**: Contiene i metadati dei file audio con colonne come:
  - `slice_file_name`: Nome del file audio.
  - `fsID`: ID della registrazione originale.
  - `start`, `end`: Timestamp di inizio e fine.
  - `salience`: Indica se il suono è in primo piano (1) o sullo sfondo (2).
  - `fold`: Numero del fold (1-10).
  - `classID`, `class`: ID numerico e nome della classe.

- **Organizzazione dei fold**:
  - I fold sono strutturati per evitare data leakage, mantenendo clip della stessa registrazione nello stesso fold.

### Procedura di preprocessing

1. **Caricamento dei metadati**:
    - Lettura del file CSV e analisi delle statistiche del dataset.

2. **Preprocessing audio**:
    - Resampling, normalizzazione e padding/trimming per uniformare i file audio.

3. **Estrazione delle feature**:
    - Calcolo di log-mel-spettrogrammi, MFCC, centroidi spettrali e roll-off.

4. **Visualizzazione**:
    - Esempio di preprocessing e visualizzazione di spettrogrammi.

5. **Elaborazione completa**:
    - Iterazione su tutti i file del dataset per preprocessarli e salvarli.

6. **Salvataggio delle feature**:
    - Estrazione e salvataggio delle feature in formato `.npz` per ogni file audio.

### Nota sulla cross-validation

- **Importanza dei fold**:
  - I fold predefiniti devono essere mantenuti per evitare leakage.
  - La cross-validation con tutti i 10 fold garantisce valutazioni affidabili.

- **Validità statistica**:
  - Ogni fold contiene clip provenienti da registrazioni diverse, assicurando che non ci sia sovrapposizione tra training e test set.

In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:


# Configurazione percorsi
DATA_ROOT = "../data"
RAW_DATA = os.path.join(DATA_ROOT, "raw")
PROCESSED_DATA = os.path.join(DATA_ROOT, "processed/step1")
META_FILE = os.path.join(RAW_DATA, "UrbanSound8K.csv")

# Parametri di preprocessing
TARGET_SR = 22050  # Frequenza di campionamento target
TARGET_LENGTH = 4  # Lunghezza audio in secondi

# Crea directory output se non esistono
os.makedirs(PROCESSED_DATA, exist_ok=True)
for i in range(1, 11):
    os.makedirs(os.path.join(PROCESSED_DATA, f"fold{i}"), exist_ok=True)

# Caricamento metadati
df = pd.read_csv(META_FILE)
print(f"Numero totale di clip: {len(df)}")

# Statistiche sul dataset
print("\nDistribuzione delle classi:")
class_dist = df['class'].value_counts()
print(class_dist)

print("\nDistribuzione dei fold:")
fold_dist = df['fold'].value_counts().sort_index()
print(fold_dist)

# Funzione di preprocessing
def preprocess_audio(audio_path, target_sr=TARGET_SR):
    """
    Applica preprocessing all'audio:
    1. Resampling a 22,050 Hz
    2. Normalizzazione a -1 dBFS 
    3. Pad/Trim a 4 secondi esatti
    """
    # Carica il file audio
    try:
        y, sr = librosa.load(audio_path, sr=None)
        
        # Resampling a 22,050 Hz
        if sr != target_sr:
            y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
        
        # Normalizzazione a -1 dBFS (evitando division by zero)
        if np.max(np.abs(y)) > 0:
            y = y / np.max(np.abs(y))
        
        # Pad o taglia a esattamente 4 secondi
        target_length = target_sr * TARGET_LENGTH
        if len(y) < target_length:
            # Usa zero-padding manuale per versioni più vecchie di librosa
            padding = np.zeros(target_length - len(y))
            y = np.concatenate([y, padding])
        else:
            # Taglia se troppo lungo
            y = y[:target_length]
        
        return y, target_sr
    
    except Exception as e:
        print(f"Errore processando {audio_path}: {str(e)}")
        return None, None

# Estrazione features
def extract_features(y, sr):
    """
    Estrae log-mel-spettrogramma e MFCC
    """
    # Log-Mel-Spettrogramma (128 bande)
    mel_spec = librosa.feature.melspectrogram(
        y=y, 
        sr=sr, 
        n_mels=128,
        hop_length=512,
        n_fft=2048
    )
    log_mel_spec = librosa.power_to_db(mel_spec)
    
    # MFCC (20 coefficienti)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    
    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    
    # Spectral Rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    
    return {
        'log_mel_spec': log_mel_spec,
        'mfcc': mfcc,
        'centroid': centroid,
        'rolloff': rolloff
    }

# Funzione per visualizzare uno spettrogramma
def plot_spectrogram(spec, title):
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(
        spec, 
        x_axis='time', 
        y_axis='mel', 
        sr=TARGET_SR,
        hop_length=512
    )
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Esempio di preprocessing su un file
def process_example():
    # Seleziona un esempio casuale
    example = df.sample(1).iloc[0]
    fold_num = example['fold']
    filename = example['slice_file_name']
    class_name = example['class']
    
    audio_path = os.path.join(RAW_DATA, f"fold{fold_num}", filename)
    print(f"Esempio: {filename}, Classe: {class_name}")
    
    # Preprocessing
    y, sr = preprocess_audio(audio_path)
    
    # Visualizza forma d'onda
    plt.figure(figsize=(10, 3))
    plt.plot(np.linspace(0, TARGET_LENGTH, len(y)), y)
    plt.title(f"Forma d'onda: {class_name}")
    plt.xlabel("Tempo (s)")
    plt.ylabel("Ampiezza")
    plt.tight_layout()
    plt.show()
    
    # Estrai e visualizza features
    features = extract_features(y, sr)
    plot_spectrogram(features['log_mel_spec'], f"Log-Mel Spettrogramma: {class_name}")
    
    return y, sr, features

# Esegui esempio
example_audio, sr, example_features = process_example()

# Preprocessing completo del dataset
def process_all_files():
    # Crea dizionario per le statistiche
    stats = {
        'processed': 0,
        'errors': 0,
        'classes_processed': {cls: 0 for cls in df['class'].unique()}
    }
    
    print("Avvio preprocessing di tutti i file...")
    
    # Loop su tutte le righe del dataframe
    for _, row in tqdm(df.iterrows(), total=len(df)):
        fold_num = row['fold']
        filename = row['slice_file_name']
        class_name = row['class']
        
        src_path = os.path.join(RAW_DATA, f"fold{fold_num}", filename)
        dst_path = os.path.join(PROCESSED_DATA, f"fold{fold_num}", filename)
        
        # Salta se il file è già stato elaborato
        if os.path.exists(dst_path):
            stats['processed'] += 1
            stats['classes_processed'][class_name] += 1
            continue
        
        # Preprocessing
        y, sr = preprocess_audio(src_path)
        
        if y is not None:
            # Salva il file elaborato
            sf.write(dst_path, y, sr)
            stats['processed'] += 1
            stats['classes_processed'][class_name] += 1
        else:
            stats['errors'] += 1
    
    print(f"Preprocessing completato: {stats['processed']} file elaborati, {stats['errors']} errori")
    print("\nFile elaborati per classe:")
    for cls, count in stats['classes_processed'].items():
        print(f"{cls}: {count}")

# Avvio preprocessing dell'intero dataset
process_all_files()

# Salvataggio delle feature
def save_features(features_path="../data/processed/features"):
    """
    Salva le feature estratte (spettrogrammi, MFCC, ecc.)
    """
    os.makedirs(features_path, exist_ok=True)
    
    print("Estrazione e salvataggio delle feature...")
    
    for fold_num in range(1, 11):
        print(f"Processando fold {fold_num}...")
        
        # Crea cartella del fold
        fold_features_path = os.path.join(features_path, f"fold{fold_num}")
        os.makedirs(fold_features_path, exist_ok=True)
        
        # Seleziona file di questo fold
        fold_files = df[df['fold'] == fold_num]
        
        # Estrai feature per ogni file
        for _, row in tqdm(fold_files.iterrows(), total=len(fold_files)):
            filename = row['slice_file_name']
            processed_path = os.path.join(PROCESSED_DATA, f"fold{fold_num}", filename)
            
            # Nome file di output (senza estensione)
            out_name = os.path.splitext(filename)[0]
            
            # Salta se le feature sono già state estratte
            if os.path.exists(os.path.join(fold_features_path, f"{out_name}.npz")):
                continue
            
            # Carica file elaborato
            try:
                y, sr = librosa.load(processed_path, sr=TARGET_SR)
                
                # Estrai features
                features = extract_features(y, sr)
                
                # Salva features
                np.savez(
                    os.path.join(fold_features_path, f"{out_name}.npz"),
                    log_mel_spec=features['log_mel_spec'],
                    mfcc=features['mfcc'],
                    centroid=features['centroid'],
                    rolloff=features['rolloff'],
                    class_id=row['classID']
                )
            except Exception as e:
                print(f"Errore nell'estrazione delle feature per {filename}: {str(e)}")

# Avvio salvataggio feature
save_features()

print("Preprocessing e estrazione feature completati!")

Numero totale di clip: 8732

Distribuzione delle classi:
class
dog_bark            1000
children_playing    1000
air_conditioner     1000
street_music        1000
jackhammer          1000
engine_idling       1000
drilling            1000
siren                929
car_horn             429
gun_shot             374
Name: count, dtype: int64

Distribuzione dei fold:
fold
1     873
2     888
3     925
4     990
5     936
6     823
7     838
8     806
9     816
10    837
Name: count, dtype: int64
Esempio: 99500-2-0-41.wav, Classe: children_playing


KeyboardInterrupt: 

Ogni file `.npz` generato nella directory `features` (e nelle sottocartelle `foldX`) è un archivio compresso contenente diversi array NumPy. Per ogni file audio originale viene creato un file `.npz` corrispondente (ad esempio, `audio1.wav` diventa `audio1.npz`). Al suo interno trovi:

- **log_mel_spec**:  
    - **Descrizione**: Log-Mel spettrogramma, una rappresentazione 2D che mostra la distribuzione dell'energia del segnale audio attraverso bande di frequenza scalate secondo la scala Mel, nel tempo.  
    - **Forma**: Array NumPy 2D di dimensioni `(n_mels, num_frames)`, dove:  
        - `n_mels = 128` (definito in `extract_features`).  
        - `num_frames ≈ 173`, calcolato da `(22050 Hz * 4 secondi) / 512`.  
        - Forma approssimativa: `(128, 173)`.

- **mfcc**:  
    - **Descrizione**: Coefficienti Cepstrali in Scala Mel (MFCC), utili per catturare le caratteristiche timbriche del suono.  
    - **Forma**: Array NumPy 2D di dimensioni `(n_mfcc, num_frames)`, dove:  
        - `n_mfcc = 20` (definito in `extract_features`).  
        - `num_frames ≈ 173`.  
        - Forma approssimativa: `(20, 173)`.

- **centroid**:  
    - **Descrizione**: Centroide spettrale, misura che indica il "centro di massa" dello spettro, correlato alla brillantezza del suono.  
    - **Forma**: Array NumPy 2D di dimensioni `(1, num_frames)` o 1D se compresso (`squeeze`).  
        - `num_frames ≈ 173`.  
        - Forma approssimativa: `(1, 173)`.

- **rolloff**:  
    - **Descrizione**: Rolloff spettrale, frequenza sotto la quale si concentra una specifica percentuale dell'energia totale dello spettro (es. 85% o 90%).  
    - **Forma**: Simile al centroide, array NumPy 2D di dimensioni `(1, num_frames)`.  
        - `num_frames ≈ 173`.  
        - Forma approssimativa: `(1, 173)`.

- **class_id**:  
    - **Descrizione**: Identificatore numerico della classe del suono (es. `0` per "air_conditioner", `1` per "car_horn", ecc.), derivato dalla colonna `classID` del file `UrbanSound8K.csv`.  
    - **Forma**: Valore scalare (array NumPy di dimensione 0 o intero).